### <span style=color:blue> Some examples of pymongo aggregation pipelines </span>

<span style=color:blue>These are all illustrated using pymongo.  A useful exercise would be to understand how to express all of these in mongosh.</span>

In [1]:
# my usual collection of package imports; not really using them in this notebook

import sys
import json
import csv
import yaml

import importlib

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime
# with the above choices, the imported datetime.time(2023,07,01) is recognized
# from datetime import date
# from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

# Create an utilities file util.py in a folder benchmarking and import it
sys.path.append('../ECS116-HELPER-FUNCTIONS/')
import util

In [2]:
# checking that my util.py file is accessible
util.hello_world()

'hello world'

## <span style=color:blue>Getting mongodb connection set up</span>

In [3]:
from pymongo import MongoClient

client = MongoClient()

# the default port for MongoDB is 27017
# could have written client = MongoClient("localhost", 27017)
#                 or client = MongoClient("mongodb://localhost:27017/")

## <span style=color:blue>Creating a small database with 2 collections</span>

<span style=color:blue>Setting up access to the test database in MongoDB, and seeing if there are collections already there </span>

In [4]:
# set variable db_test to hold the test database in MongoDB, 
#    or create it if it doesn't already exist
db_test = client.test

db_test.list_collection_names()

['prices', 'inv_items_with_agg_info', 'inventory']

<span style=color:blue>Flushing out the test database</span>

In [5]:
# dropping the collections to have a fresh start
db_test.inventory.drop()
db_test.prices.drop()

print(db_test.list_collection_names())

['inv_items_with_agg_info']


<span style=color:blue>Creating two collections in test database, and loading some data

In [6]:
# set up for creation of inventory and price_list collections in db_test
inv = db_test.inventory
pr = db_test.prices

# A collection is not actually created until it has at least one document
print()
print(db_test.list_collection_names())


inv_list = [{ "item": "journal", "qty": 25, "size": { "h": 14, "w": 21, "uom": "cm" }, "loc": "NY" },
            { "item": "journal", "qty": 50, "size": { "h": 14, "w": 21, "uom": "cm" }, "loc": "LA" },
            { "item": "notebook", "qty": 80, "size": { "h": 8.5, "w": 11, "uom": "in" }, "loc": "NY" },
            { "item": "notebook", "qty": 20, "size": { "h": 8.5, "w": 11, "uom": "in" }, "loc": "LA" },
            { "item": "notebook", "qty": 30, "size": { "h": 8.5, "w": 11, "uom": "in" }, "loc": "SF" },
            { "item": "bottle", "qty": 30, "size": { "h": 4, "w": 10, "uom": "in" }, "loc": "NY" },
            { "item": "bottle", "qty": 40, "size": { "h": 4, "w": 10, "uom": "in" }, "loc": "SF" },
            { "item": "paper", "qty": 100, "size": { "h": 8.5, "w": 11, "uom": "in" }, "loc": "NY" },
            { "item": "paper", "qty": 120, "size": { "h": 8.5, "w": 11, "uom": "in" }, "loc": "SF" },
            { "item": "planner", "qty": 75, "size": { "h": 22.85, "w": 30, "uom": "cm" }, "loc": "LA" }
            ]

price_list = [{"descrip": "journal", "price": 9.50},
              {"descrip": "notebook", "price": 7.44},
              {"descrip": "envelopes", "price": 6.75}
             ]

# bulk inserts in pymongo
inv.insert_many(inv_list)
pr.insert_many(price_list)

print()
print(db_test.list_collection_names())

print()
# if no condition, then find returns everything
invDocs = inv.find()
for doc in invDocs:
    pprint.pp(doc)

print()
prDocs = pr.find()
for doc in prDocs:
    pprint.pp(doc)


['inv_items_with_agg_info']

['prices', 'inventory', 'inv_items_with_agg_info']

{'_id': ObjectId('683c8be15d9ea36338203c27'),
 'item': 'journal',
 'qty': 25,
 'size': {'h': 14, 'w': 21, 'uom': 'cm'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c28'),
 'item': 'journal',
 'qty': 50,
 'size': {'h': 14, 'w': 21, 'uom': 'cm'},
 'loc': 'LA'}
{'_id': ObjectId('683c8be15d9ea36338203c29'),
 'item': 'notebook',
 'qty': 80,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c2a'),
 'item': 'notebook',
 'qty': 20,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'LA'}
{'_id': ObjectId('683c8be15d9ea36338203c2b'),
 'item': 'notebook',
 'qty': 30,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'SF'}
{'_id': ObjectId('683c8be15d9ea36338203c2c'),
 'item': 'bottle',
 'qty': 30,
 'size': {'h': 4, 'w': 10, 'uom': 'in'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c2d'),
 'item': 'bottle',
 'qty': 40,
 'size': {'h': 4, 'w': 10, 'uom':

### <span style=color:blue>Example of $lookup (variant on left join).</span>

In [7]:
# We will be doing a kind of left join, which will produce 
#     ALL documents from prices
#     For SOME (or maybe ALL) of those documents, data from related inventory documents will be "glued into"
#           the price document as a subtree somewhere

# First, we specify a "pipeline" specification, which identifies how to make the join and what data 
#     from the inventory document is to be included
pipeline = [
    { 
        '$lookup': {                  # this is the operator name at the heart of this kind of left join

            'from': 'inventory',        # need collection name here; do not use 
                                        #      "inv", which is variable holding collection
            'localField': 'descrip',    # field name in inventory for doing the match 
                                        #      (this is "local" to prices, which is what
                                        #        this pipeline is going to run on)
            'foreignField': 'item',     # field name in inventory for doing the match
                                        #      (this is "foriegn" to prices, which
                                        #       which the pipeline is running on)
            'as': 'invinfo'
        }
    }
]
for doc in pr.aggregate(pipeline):         # the "aggregate" function is actually creating the variant of left join
    pprint.pp(doc)

{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'invinfo': [{'_id': ObjectId('683c8be15d9ea36338203c27'),
              'item': 'journal',
              'qty': 25,
              'size': {'h': 14, 'w': 21, 'uom': 'cm'},
              'loc': 'NY'},
             {'_id': ObjectId('683c8be15d9ea36338203c28'),
              'item': 'journal',
              'qty': 50,
              'size': {'h': 14, 'w': 21, 'uom': 'cm'},
              'loc': 'LA'}]}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'invinfo': [{'_id': ObjectId('683c8be15d9ea36338203c29'),
              'item': 'notebook',
              'qty': 80,
              'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
              'loc': 'NY'},
             {'_id': ObjectId('683c8be15d9ea36338203c2a'),
              'item': 'notebook',
              'qty': 20,
              'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
              'loc': 'LA'},
             {'_id': Obje

### <span style=color:blue>Example of $unwind, to "flatten" the previous output</span>

In [8]:
pipeline = [
    { 
        '$lookup': {                  
            'from': 'inventory',       
            'localField': 'descrip',    
            'foreignField': 'item',     
            'as': 'invinfo'
        }
    },
    {
        '$unwind' : { 
            'path': '$invinfo'
        }
    }
]

result = pr.aggregate(pipeline)

for doc in result:
    pprint.pp(doc)

{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c27'),
             'item': 'journal',
             'qty': 25,
             'size': {'h': 14, 'w': 21, 'uom': 'cm'},
             'loc': 'NY'}}
{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c28'),
             'item': 'journal',
             'qty': 50,
             'size': {'h': 14, 'w': 21, 'uom': 'cm'},
             'loc': 'LA'}}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c29'),
             'item': 'notebook',
             'qty': 80,
             'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
             'loc': 'NY'}}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c2a'),
             'i

### <span style=color:blue>What happened to envelopes?  It had an empty list for invinfo.  To include that in our result we can use the 'preserveNullAndEmptyArrays' parameter</span>

In [9]:
pipeline = [
    { 
        '$lookup': {                  
            'from': 'inventory',       
            'localField': 'descrip',    
            'foreignField': 'item',     
            'as': 'invinfo'
        }
    },
    {
        '$unwind' : { 
            'path': '$invinfo',
            'preserveNullAndEmptyArrays': True
        }
    }
]

result = pr.aggregate(pipeline)

for doc in result:
    pprint.pp(doc)

{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c27'),
             'item': 'journal',
             'qty': 25,
             'size': {'h': 14, 'w': 21, 'uom': 'cm'},
             'loc': 'NY'}}
{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c28'),
             'item': 'journal',
             'qty': 50,
             'size': {'h': 14, 'w': 21, 'uom': 'cm'},
             'loc': 'LA'}}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c29'),
             'item': 'notebook',
             'qty': 80,
             'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
             'loc': 'NY'}}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'invinfo': {'_id': ObjectId('683c8be15d9ea36338203c2a'),
             'i

### <span style=color:blue>Pulling some values of an inner list to the top level, and dropping the invinfo array</span>

In [10]:
pipeline = [
    { 
        '$lookup': {                  
            'from': 'inventory',       
            'localField': 'descrip',    
            'foreignField': 'item',     
            'as': 'invinfo'
        }
    },
    {
        '$unwind' : { 
            'path': '$invinfo',
            'preserveNullAndEmptyArrays': True
        }
    },
    {
        '$addFields': {
            'warehouse_location': '$$ROOT.invinfo.loc',
            'quantity_at_warehouse': '$$ROOT.invinfo.qty'
        }
    },
    {
        '$unset': 'invinfo'
    }
]

result = pr.aggregate(pipeline)

for doc in result:
    pprint.pp(doc)

{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'warehouse_location': 'NY',
 'quantity_at_warehouse': 25}
{'_id': ObjectId('683c8be15d9ea36338203c31'),
 'descrip': 'journal',
 'price': 9.5,
 'warehouse_location': 'LA',
 'quantity_at_warehouse': 50}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'warehouse_location': 'NY',
 'quantity_at_warehouse': 80}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'warehouse_location': 'LA',
 'quantity_at_warehouse': 20}
{'_id': ObjectId('683c8be15d9ea36338203c32'),
 'descrip': 'notebook',
 'price': 7.44,
 'warehouse_location': 'SF',
 'quantity_at_warehouse': 30}
{'_id': ObjectId('683c8be15d9ea36338203c33'),
 'descrip': 'envelopes',
 'price': 6.75}


### <span style=color:blue>Example of match (kind of selection) and group (a variation on SQL group by); also writing the output into a collection </span>

<span style=color:blue>First, printing contents of inv_list, so that we can easily look at its elements</span>

In [11]:
for doc in inv.find():
    pprint.pp(doc)

{'_id': ObjectId('683c8be15d9ea36338203c27'),
 'item': 'journal',
 'qty': 25,
 'size': {'h': 14, 'w': 21, 'uom': 'cm'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c28'),
 'item': 'journal',
 'qty': 50,
 'size': {'h': 14, 'w': 21, 'uom': 'cm'},
 'loc': 'LA'}
{'_id': ObjectId('683c8be15d9ea36338203c29'),
 'item': 'notebook',
 'qty': 80,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c2a'),
 'item': 'notebook',
 'qty': 20,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'LA'}
{'_id': ObjectId('683c8be15d9ea36338203c2b'),
 'item': 'notebook',
 'qty': 30,
 'size': {'h': 8.5, 'w': 11, 'uom': 'in'},
 'loc': 'SF'}
{'_id': ObjectId('683c8be15d9ea36338203c2c'),
 'item': 'bottle',
 'qty': 30,
 'size': {'h': 4, 'w': 10, 'uom': 'in'},
 'loc': 'NY'}
{'_id': ObjectId('683c8be15d9ea36338203c2d'),
 'item': 'bottle',
 'qty': 40,
 'size': {'h': 4, 'w': 10, 'uom': 'in'},
 'loc': 'SF'}
{'_id': ObjectId('683c8be15d9ea36338203c2e'),
 'item': 'pape

<span style=color:blue>Example of '\\$match' (a selection operator), and '\$group'</span>

In [10]:
# grouping inv_list based on the item values

# Note the use of '$' before attribute names of inv_list.
#   What happens if you leave out the '$'


pipeline = [
    {
        '$match' : { 
            'qty' : {
                '$gte' : 25 
            }
        }
    },
    {
        '$group' : {
            '_id': '$item',
            'average_quantity': {'$avg': '$qty'},
            'locations': { '$push' : '$loc' }      # $push creates a list of the
                                                   # values in the loc field
        }
    }
]
for doc in inv.aggregate(pipeline):         # the "aggregate" function is actually creating the variant of left join
    pprint.pp(doc)


{'_id': 'journal', 'average_quantity': 37.5, 'locations': ['NY', 'LA']}
{'_id': 'bottle', 'average_quantity': 35.0, 'locations': ['NY', 'SF']}
{'_id': 'notebook', 'average_quantity': 55.0, 'locations': ['NY', 'SF']}
{'_id': 'paper', 'average_quantity': 110.0, 'locations': ['NY', 'SF']}
{'_id': 'planner', 'average_quantity': 75.0, 'locations': ['LA']}


<span style=color:blue>Adding a clause to write the output to a collection</span>

In [11]:
pipeline = [
    {
        '$match' : { 
            'qty' : {
                '$gte' : 25 
            }
        }
    },
    {
        '$group' : {
            '_id': '$item',
            'average_quantity': {'$avg': '$qty'},
            'locations': { '$push' : '$loc' }      
        }
    },
    {
        '$out' : 'inv_items_with_agg_info'    # this overwrites the collection
                                              # with this name, if there is one
    }
]

result = inv.aggregate(pipeline)

 

for doc in db_test.inv_items_with_agg_info.find({}):
    pprint.pp(doc)

{'_id': 'notebook', 'average_quantity': 55.0, 'locations': ['NY', 'SF']}
{'_id': 'bottle', 'average_quantity': 35.0, 'locations': ['NY', 'SF']}
{'_id': 'planner', 'average_quantity': 75.0, 'locations': ['LA']}
{'_id': 'journal', 'average_quantity': 37.5, 'locations': ['NY', 'LA']}
{'_id': 'paper', 'average_quantity': 110.0, 'locations': ['NY', 'SF']}
